<a href="https://colab.research.google.com/github/AliAI11/DeepLens/blob/main/notebooks/02_train_vit_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# installing packages
!pip install -q transformers torch torchvision pillow tqdm scikit-learn pandas numpy huggingface_hub


In [ ]:
# imports and seeds
import os
import json
import zipfile
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import ViTForImageClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from huggingface_hub import snapshot_download
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
print(f'gpu: {torch.cuda.get_device_name(0) if device == "cuda" else "n/a"}')

device: cuda
gpu: NVIDIA A100-SXM4-40GB


In [ ]:
# upload metadata from notebook 01
from google.colab import files

os.makedirs('./data', exist_ok=True)

print('upload splits.csv and dataset_config.json:')
uploaded = files.upload()
for fname in uploaded.keys():
    os.rename(fname, f'./data/{fname}')

splits_df = pd.read_csv('./data/splits.csv')
with open('./data/dataset_config.json') as f:
    ds_config = json.load(f)

print(f'\nloaded {len(splits_df)} rows')
print(splits_df['split'].value_counts())

upload splits.csv and dataset_config.json:


Saving dataset_config.json to dataset_config.json
Saving splits.csv to splits.csv

loaded 25000 rows
split
train           20000
val_internal     2500
test             2500
Name: count, dtype: int64


In [ ]:
# re-download and extract shard_0
os.makedirs('./ntire_train', exist_ok=True)

print('downloading shard_0.zip...')
snapshot_download(
    repo_id="deepfakesMSU/NTIRE-RobustAIGenDetection-train",
    repo_type="dataset",
    allow_patterns=["shard_0.zip"],
    local_dir="./ntire_train"
)

print('extracting...')
with zipfile.ZipFile('./ntire_train/shard_0.zip', 'r') as z:
    z.extractall('./ntire_train')
os.remove('./ntire_train/shard_0.zip')

images_dir = './ntire_train/shard_0/images'
print(f'{len(os.listdir(images_dir))} images extracted')

downloading shard_0.zip...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

extracting...
50000 images extracted


In [ ]:
# PyTorch dataset
class DeepfakeDataset(Dataset):
    """loads images by image_name from labels.csv index"""
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, row['image_name'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

In [ ]:
# transforms and dataloaders
# vit-base-patch16-224 normalization
mean, std = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_df = splits_df[splits_df['split'] == 'train']
val_df = splits_df[splits_df['split'] == 'val_internal']
test_df = splits_df[splits_df['split'] == 'test']

train_ds = DeepfakeDataset(train_df, images_dir, train_transform)
val_ds = DeepfakeDataset(val_df, images_dir, eval_transform)
test_ds = DeepfakeDataset(test_df, images_dir, eval_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f'train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

train: 20000, val: 2500, test: 2500


In [ ]:
print('loading vit-base-patch16-224...')
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=2,
    ignore_mismatched_sizes=True
).to(device)

print(f'parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

loading vit-base-patch16-224...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


parameters: 85.8M


In [ ]:
# optimizer + scheduler
num_epochs = 5
total_steps = len(train_loader) * num_epochs
warmup_steps = int(total_steps * 0.1)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.CrossEntropyLoss()

print(f'epochs: {num_epochs}, total steps: {total_steps}, warmup: {warmup_steps}')

epochs: 5, total steps: 1565, warmup: 156


In [ ]:
# training
def evaluate(model, loader):
    model.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(pixel_values=x).logits
            p = torch.softmax(logits.float(), dim=1)
            preds.extend(p.argmax(dim=1).cpu().numpy())
            probs.extend(p[:, 1].cpu().numpy())
            labels.extend(y.numpy())
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds),
        'auroc': roc_auc_score(labels, probs),
        'preds': preds, 'probs': probs, 'labels': labels
    }

best_val_acc = 0.0
os.makedirs('./models', exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'epoch {epoch+1}/{num_epochs}')
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(pixel_values=x).logits
            loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    m = evaluate(model, val_loader)
    print(f'epoch {epoch+1}: train_loss={train_loss/len(train_loader):.4f}, '
          f'val_acc={m["accuracy"]:.4f}, val_f1={m["f1"]:.4f}, val_auroc={m["auroc"]:.4f}')

    if m['accuracy'] > best_val_acc:
        best_val_acc = m['accuracy']
        model.save_pretrained('./models/vit_deepfake')
        print(f'  saved best (val_acc={best_val_acc:.4f})')

print(f'\nbest val accuracy: {best_val_acc:.4f}')

epoch 1/5: 100%|██████████| 313/313 [02:25<00:00,  2.15it/s, loss=0.2445]


epoch 1: train_loss=0.4699, val_acc=0.8932, val_f1=0.8921, val_auroc=0.9687


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val_acc=0.8932)


epoch 2/5: 100%|██████████| 313/313 [02:13<00:00,  2.34it/s, loss=0.1096]


...

epoch 4: train_loss=0.0316, val_acc=0.9380, val_f1=0.9392, val_auroc=0.9851


epoch 5/5: 100%|██████████| 313/313 [02:29<00:00,  2.09it/s, loss=0.0217]


epoch 5: train_loss=0.0172, val_acc=0.9404, val_f1=0.9420, val_auroc=0.9852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val_acc=0.9404)

best val accuracy: 0.9404


In [ ]:
# test evaluation
model = ViTForImageClassification.from_pretrained('./models/vit_deepfake').to(device)

m = evaluate(model, test_loader)
print(f'test accuracy:  {m["accuracy"]:.4f}')
print(f'test f1:        {m["f1"]:.4f}')
print(f'test auroc:     {m["auroc"]:.4f}')
print(f'test precision: {precision_score(m["labels"], m["preds"]):.4f}')
print(f'test recall:    {recall_score(m["labels"], m["preds"]):.4f}')

test_results = test_df.copy().reset_index(drop=True)
test_results['pred_label'] = m['preds']
test_results['pred_prob_fake'] = m['probs']
test_results.to_csv('./data/test_predictions.csv', index=False)
print(f'\nsaved ./data/test_predictions.csv')

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

test accuracy:  0.9284
test f1:        0.9288
test auroc:     0.9824
test precision: 0.9262
test recall:    0.9314

saved ./data/test_predictions.csv


In [ ]:
# reload model with eager attention so output_attentions actually works
print('reloading model with eager attention...')
model = ViTForImageClassification.from_pretrained(
    './models/vit_deepfake',
    attn_implementation='eager'
).to(device)
model.eval()

def extract_attention(model, loader):
    """last-layer cls-to-patch attention, averaged over heads, reshaped to 14x14"""
    all_maps = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc='extracting attention'):
            x = x.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                out = model(pixel_values=x, output_attentions=True)
            attn = out.attentions[-1].float().mean(dim=1)  # [B, 197, 197]
            cls_attn = attn[:, 0, 1:]                       # [B, 196]
            maps = cls_attn.reshape(-1, 14, 14).cpu().numpy()
            all_maps.append(maps)
    return np.concatenate(all_maps, axis=0)

attention_maps = extract_attention(model, test_loader)
print(f'attention maps shape: {attention_maps.shape}')
np.save('./data/test_attention_maps.npy', attention_maps)
print('saved ./data/test_attention_maps.npy')

reloading model with eager attention...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

extracting attention: 100%|██████████| 20/20 [00:19<00:00,  1.04it/s]

attention maps shape: (2500, 14, 14)
saved ./data/test_attention_maps.npy


In [ ]:
print('zipping model and downloading...')
shutil.make_archive('vit_deepfake', 'zip', './models/vit_deepfake')

files.download('vit_deepfake.zip')
files.download('./data/test_predictions.csv')
files.download('./data/test_attention_maps.npy')

zipping model and downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>